# 06 — Interactive Click Applications

> We are no longer drawing maps.
> We are asking maps questions.

Everything in this module has been building to this cell:

```text
User clicks
  → ipyleaflet gives us [lat, lon]
  → we flip to (lon, lat)
  → ray casting tests all polygons
  → we extract properties
  → we display a human-readable answer
```

That is the full pipeline. Each piece is something you've already built. This notebook assembles them.

## Setup — All Tools in One Place

In [ ]:
import json
import math
from pathlib import Path
from ipyleaflet import Map, GeoJSON, Marker, LayersControl, basemaps
from ipywidgets import Output

DATA_DIR = Path("data")

with open(DATA_DIR / "regions.geojson") as f:
    regions_fc = json.load(f)


# --- Geometry ---

def point_in_ring(lon, lat, ring):
    n = len(ring) - 1
    inside = False
    for i in range(n):
        xi, yi = ring[i][0],   ring[i][1]
        xj, yj = ring[i+1][0], ring[i+1][1]
        if (yi > lat) != (yj > lat):
            x_intersect = xi + (lat - yi) * (xj - xi) / (yj - yi)
            if lon < x_intersect:
                inside = not inside
    return inside

def point_in_feature(lon, lat, feature):
    geom = feature["geometry"]
    if geom["type"] != "Polygon":
        return False
    return point_in_ring(lon, lat, geom["coordinates"][0])

def classify_point(lon, lat, feature_collection):
    for feature in feature_collection["features"]:
        if point_in_feature(lon, lat, feature):
            return {"found": True, "lon": lon, "lat": lat,
                    "name": feature["properties"].get("name"),
                    "properties": feature["properties"],
                    "feature": feature}
    return {"found": False, "lon": lon, "lat": lat,
            "name": None, "properties": {}, "feature": None}


# --- Buffer (from module 10) ---

def destination_point(lon, lat, bearing_deg, distance_km):
    R = 6371.0; d = distance_km / R
    phi1 = math.radians(lat); lam1 = math.radians(lon)
    theta = math.radians(bearing_deg)
    phi2 = math.asin(math.sin(phi1)*math.cos(d) + math.cos(phi1)*math.sin(d)*math.cos(theta))
    lam2 = lam1 + math.atan2(math.sin(theta)*math.sin(d)*math.cos(phi1),
                              math.cos(d) - math.sin(phi1)*math.sin(phi2))
    return [math.degrees(lam2), math.degrees(phi2)]

def haversine_km(lon1, lat1, lon2, lat2):
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1); dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return R * 2 * math.asin(math.sqrt(a))

def make_circle(lon, lat, radius_km, n=64):
    ring = [destination_point(lon, lat, 360.0*i/n, radius_km) for i in range(n)]
    ring.append(ring[0])
    return {"type": "Polygon", "coordinates": [ring]}


SECTOR_COLORS = {
    "Sector Alpha":   "#e74c3c",
    "Sector Bravo":   "#e67e22",
    "Sector Charlie": "#2980b9",
    "Sector Delta":   "#27ae60",
    "Sector Echo":    "#8e44ad",
}

print("All tools loaded.")

## The Complete Pipeline

One click handler, all five stages in order.

In [ ]:
placed_markers = []
out = Output()

m = Map(center=(28, 45), zoom=3, basemap=basemaps.CartoDB.Positron)

# Draw sectors
for feat in regions_fc["features"]:
    color = SECTOR_COLORS.get(feat["properties"]["name"], "#95a5a6")
    m.add(GeoJSON(
        data={"type": "FeatureCollection", "features": [feat]},
        style={"color": color, "fillColor": color, "fillOpacity": 0.12, "weight": 2}
    ))

def on_click(**kwargs):
    if kwargs.get('type') != 'click':
        return

    # Stage 1: receive click (ipyleaflet gives [lat, lon])
    click_coords = kwargs['coordinates']
    lat, lon = click_coords[0], click_coords[1]     # stage 2: flip

    # Stage 3: classify
    result = classify_point(lon, lat, regions_fc)   # stage 3: ray cast

    # Stage 4: display
    # Remove previous marker
    for old in placed_markers:
        m.remove(old)
    placed_markers.clear()

    marker = Marker(location=(lat, lon))
    m.add(marker)
    placed_markers.append(marker)

    with out:
        out.clear_output(wait=True)
        print(f"Click at  lat={lat:.4f}  lon={lon:.4f}")
        if result["found"]:
            props = result["properties"]
            print(f"  Region:  {props.get('name', '—')}")
            print(f"  Detail:  {props.get('description', '—')}")
        else:
            print("  Outside all sectors.")

m.on_interaction(on_click)
display(m, out)

## Highlighting the Clicked Region

Instead of just printing, we can highlight the matched polygon on the map by updating its GeoJSON style dynamically. This requires rebuilding the layer when a new click arrives.

In [ ]:
active_layers = []
out2 = Output()

m2 = Map(center=(28, 45), zoom=3, basemap=basemaps.CartoDB.Positron)

# Add all sectors as base layers (dim)
sector_layers = {}
for feat in regions_fc["features"]:
    color = SECTOR_COLORS.get(feat["properties"]["name"], "#95a5a6")
    layer = GeoJSON(
        data={"type": "FeatureCollection", "features": [feat]},
        style={"color": color, "fillColor": color, "fillOpacity": 0.10, "weight": 1.5}
    )
    sector_layers[feat["properties"]["name"]] = {"layer": layer, "color": color, "feat": feat}
    m2.add(layer)

def on_click_highlight(**kwargs):
    if kwargs.get('type') != 'click':
        return

    lat, lon = kwargs['coordinates'][0], kwargs['coordinates'][1]
    result = classify_point(lon, lat, regions_fc)

    # Remove any highlight layers from previous click
    for layer in active_layers:
        m2.remove(layer)
    active_layers.clear()

    if result["found"]:
        sector_name = result["name"]
        color = SECTOR_COLORS.get(sector_name, "#e74c3c")

        # Add a bright highlight layer on top
        highlight = GeoJSON(
            data={"type": "FeatureCollection", "features": [result["feature"]]},
            style={"color": color, "fillColor": color, "fillOpacity": 0.55, "weight": 3}
        )
        m2.add(highlight)
        active_layers.append(highlight)

    # Add click marker
    marker = Marker(location=(lat, lon))
    m2.add(marker)
    active_layers.append(marker)

    with out2:
        out2.clear_output(wait=True)
        name = result["name"] or "(outside all sectors)"
        print(f"Clicked: {name}")

m2.on_interaction(on_click_highlight)
display(m2, out2)

## Application: Is This Click Inside a Blast Zone?

The ray casting algorithm works on any polygon — including the circular blast zones we built in module 10. Here we connect the two modules: click anywhere on the map and instantly know whether that location is within the danger radius of each target.

In [ ]:
# Build blast zone polygons around the module 10 targets
targets = [
    {"name": "Tehran",  "lon": 51.388, "lat": 35.695, "color": "#e74c3c"},
    {"name": "Riyadh",  "lon": 46.675, "lat": 24.688, "color": "#e67e22"},
    {"name": "Cairo",   "lon": 31.235, "lat": 30.044, "color": "#2980b9"},
]
BLAST_RADIUS_KM = 800

blast_fc = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {"name": f"{t['name']} {BLAST_RADIUS_KM} km zone",
                           "target": t["name"],
                           "radius_km": BLAST_RADIUS_KM},
            "geometry": make_circle(t["lon"], t["lat"], BLAST_RADIUS_KM)
        }
        for t in targets
    ]
}

out3 = Output()
m3 = Map(center=(30, 42), zoom=3, basemap=basemaps.CartoDB.Positron)

# Draw blast zones
for feat, t in zip(blast_fc["features"], targets):
    m3.add(GeoJSON(
        data={"type": "FeatureCollection", "features": [feat]},
        name=feat["properties"]["name"],
        style={"color": t["color"], "fillColor": t["color"],
               "fillOpacity": 0.12, "weight": 2}
    ))
m3.add(LayersControl())

active3 = []

def on_blast_click(**kwargs):
    if kwargs.get('type') != 'click':
        return

    lat, lon = kwargs['coordinates'][0], kwargs['coordinates'][1]

    # Find ALL blast zones containing this point
    matches = [
        feat for feat in blast_fc["features"]
        if point_in_feature(lon, lat, feat)
    ]

    for layer in active3:
        m3.remove(layer)
    active3.clear()
    marker = Marker(location=(lat, lon))
    m3.add(marker)
    active3.append(marker)

    with out3:
        out3.clear_output(wait=True)
        print(f"Click at lat={lat:.3f}  lon={lon:.3f}")
        if matches:
            print(f"  Within {len(matches)} blast zone(s):")
            for m in matches:
                props = m["properties"]
                tgt = next(t for t in targets if t["name"] == props["target"])
                dist = haversine_km(lon, lat, tgt["lon"], tgt["lat"])
                print(f"    ✓ {props['name']}  ({dist:.0f} km from {props['target']})")
        else:
            print("  Outside all blast zones.")

m3.on_interaction(on_blast_click)
display(m3, out3)

## The Door to Real GIS

We've been working with polygons defined as flat `[lon, lat]` lists and testing containment with a Cartesian ray cast. This works well at small scales and low latitudes.

When it breaks:

> **What if the polygon is curved on Earth?**

Geographic polygons at large scale or high latitude have edges that follow great circles — curves on a sphere, not straight lines in lon/lat space. Our ray casting algorithm treats all edges as straight lines in degree space, which introduces error proportional to the polygon's geographic extent and latitude.

This is exactly the CRS limitation from module 10, applied to polygon containment. The solution is **projection** — transforming the sphere's surface onto a flat plane before running Cartesian geometry. That's what GIS tools like Shapely, GeoPandas, and PostGIS do internally.

What we've built here is the foundation: you now understand what those tools are doing when they answer a point-in-polygon query.

## Exercise A

Build a **region classifier map** that:

1. Shows all five sectors.
2. On each click, prints the sector name and description in the output.
3. Places a Marker at the click location with a different color depending on whether the click was inside or outside a sector.
4. Accumulates all clicks — does not clear previous markers.

In [ ]:
from ipyleaflet import Map, GeoJSON, Marker, basemaps
from ipywidgets import Output
import json
from pathlib import Path

DATA_DIR = Path("data")
with open(DATA_DIR / "regions.geojson") as f:
    regions_fc = json.load(f)

out = Output()

# Map showing all sectors
# Click handler: classify, print result, add persistent marker
# Inside click: one marker color; outside click: different color
# Your code here

## Exercise B

Build a **blast zone query map** that answers: *"Which targets can reach this location?"*

1. Draw three concentric rings (200 km, 500 km, 1000 km) around **Tehran** and **Riyadh** — six rings total.
2. On each click, determine which rings contain the click point using `point_in_ring`.
3. Print a summary: for each target, report the closest ring that contains the click (or "out of range").
4. Example output:
   ```
   Click at lat=28.5  lon=50.0
   Tehran:  within 500 km zone (432 km away)
   Riyadh:  within 200 km zone (187 km away)
   ```

In [ ]:
from ipyleaflet import Map, GeoJSON, LayersControl, basemaps
from ipywidgets import Output
import math

# Target coordinates
tehran = (51.388, 35.695)
riyadh = (46.675, 24.688)

# 1. Three concentric rings per target (200, 500, 1000 km)
# 2. Click handler: determine which rings contain the click
# 3. For each target: report closest containing ring, distance
# 4. Display on map with LayersControl
# Your code here

---

## Check Your Understanding

**1.** Trace the full pipeline from a user click to a displayed region name. List every step and what data transformation happens at each.

**2.** We treated polygon edges as straight lines in degree space. Under what geographic conditions would this produce the most error, and what would you need to do to fix it?

```python
# No code needed — answer in your own words
```

## Next

In [12 — Refactoring](../../12-Refactoring/00-Introduction.ipynb), we take the functions we've been redefining across every notebook — `destination_point`, `haversine_km`, `point_in_ring`, `classify_point` — and organize them into a reusable module, so future notebooks import rather than reimplement.